In [0]:
%pip install torch
%pip install pyrosetta-installer
!python -c 'import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()'
%pip install biopython
%pip install tqdm
%pip install dm-tree
%pip install biotite
%pip install pyyaml
#!pip install easydict
#!pip install modelcif

working_dir = "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation"

!git clone https://gitlab.com/mjslee0921/flowpacker
%pip install --use-deprecated=legacy-resolver -r flowpacker/requirements.txt

#!git clone https://github.com/cytokineking/ProtenixScore
#%cd /Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProtenixScore
#!./install_protenixscore.sh --no-conda

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Installing PyRosetta:
 os: ubuntu
 type: Release
 Rosetta C++ extras: 
 mirror: https://west.rosettacommons.org/pyrosetta/release/release
 extra packages: numpy

PyRosetta wheel url: https://:@west.rosettacommons.org/pyrosetta/release/release/PyRosetta4.Release.python312.ubuntu.wheel/pyrosetta-2026.6+release.e5a76a2dbd-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB ? eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.9 MB/s eta 0:00:00
Note: you may need to restart 

In [0]:
#@title 1. MPNN Squence Design
import subprocess
import sys

script = f"{working_dir}/ProteinMPNN/protein_mpnn_run.py"
folder_with_pdbs = f"{working_dir}test/"
path_for_fixed_positions = f"{working_dir}test/fixed_positions.jsonl"
path_for_parsed_chains = f"{working_dir}test/parsed_pdbs.jsonl"
chains_to_design ="A"
num_seq_per_target=8
sampling_temp =0.5
seed = 37
fixed_positions="1 2 3 4 5" #fixing/not designing residues 1 2 3...25 in chain A and residues 10 11 12...40 in chain C

# setup
!python $working_dir/ProteinMPNN/helper_scripts/parse_multiple_chains.py --input_path=$folder_with_pdbs --output_path=$path_for_parsed_chains
!python $working_dir/ProteinMPNN/helper_scripts/assign_fixed_chains.py --input_path=$path_for_parsed_chains --output_path=$path_for_assigned_chains --chain_list "$chains_to_design"
!python $working_dir/ProteinMPNN/helper_scripts/make_fixed_positions_dict.py --input_path=$path_for_parsed_chains --output_path=$path_for_fixed_positions --chain_list "$chains_to_design" --position_list "$fixed_positions"

# Run actual protein_mpnn_run script
subprocess.run([sys.executable, script,
                "--use_soluble_model",
                "--folder_with_pdbs", folder_with_pdbs, 
                "--out_folder", out_dir, 
                "--chain_list", chains_to_design, 
                "--fixed_positions_jsonl", path_for_fixed_positions, 
                "--num_seq_per_target", str(num_seq_per_target), 
                "--sampling_temp", str(sampling_temp), 
                "--seed", str(seed), 
                "--batch_size", str(num_seq_per_target),
                "--omit_AAs", "C"])

# MPNN with AA bias
bias_AA_jsonl =f"{working_dir}bias_AA.jsonl"

subprocess.run([sys.executable, script,
                "--folder_with_pdbs", folder_with_pdbs, 
                "--out_folder", out_dir, 
                "--chain_list", chains_to_design, 
                "--fixed_positions_jsonl", path_for_fixed_positions, 
                "--num_seq_per_target", str(num_seq_per_target), 
                "--sampling_temp", str(sampling_temp), 
                "--seed", str(seed), 
                "--omit_AAs", "C"])


Traceback (most recent call last):
  File "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProteinMPNN/helper_scripts/parse_multiple_chains.py", line 310, in <module>
    main(args)
  File "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProteinMPNN/helper_scripts/parse_multiple_chains.py", line 297, in main
    with open(save_path, 'w') as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturationtest/parsed_pdbs.jsonl'
python: can't open file '/ProteinMPNN/helper_scripts/assign_fixed_chains.py': [Errno 2] No such file or directory
Traceback (most recent call last):
  File "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProteinMPNN/helper_scripts/make_fixed_positions_dict.py", line 75, in <module>
    main(args)
  File "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProteinMPNN/helper_scripts/make

Traceback (most recent call last):
  File "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProteinMPNN/protein_mpnn_run.py", line 488, in <module>
    main(args)   
    ^^^^^^^^^^
  File "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProteinMPNN/protein_mpnn_run.py", line 31, in main
    print("fixed_positions_dict: {}".format(fixed_positions_dict))
                                            ^^^^^^^^^^^^^^^^^^^^
UnboundLocalError: cannot access local variable 'fixed_positions_dict' where it is not associated with a value
Traceback (most recent call last):
  File "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProteinMPNN/protein_mpnn_run.py", line 488, in <module>
    main(args)   
    ^^^^^^^^^^
  File "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProteinMPNN/protein_mpnn_run.py", line 31, in main
    print("fixed_positions_dict: {}".format(fixed_positions_dict))
                            

CompletedProcess(args=['/local_disk0/.ephemeral_nfs/envs/pythonEnv-16c7f3e3-9308-473d-b68f-561c46f53da4/bin/python', '/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/ProteinMPNN/protein_mpnn_run.py', '--folder_with_pdbs', '/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturationtest/', '--out_folder', '/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/test/mpnn_out/', '--chain_list', 'A', '--fixed_positions_jsonl', '/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturationtest/fixed_positions.jsonl', '--num_seq_per_target', '8', '--sampling_temp', '0.5', '--seed', '37', '--omit_AAs', 'C'], returncode=1)

In [0]:
#@title 2. Run flowpacker
#%cd /Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/flowpacker/
#!python $working_dir/sampler_pdb_colab.py base --pdb_dir $working_dir/test/ --save_dir $working_dir/test/out/

%cd flowpacker/
!python /Workspace/Users/david.nedrud/Documents/GitHub/MPNN_affinity_maturation/sampler_pdb_colab.py base --pdb_dir /Workspace/Users/david.nedrud/Documents/GitHub/MPNN_affinity_maturation/test/mpnn_out --save_dir /Workspace/Users/david.nedrud/Documents/GitHub/MPNN_affinity_maturation/test/out/


/opt/miniconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/david.nedrud/Documents/GitHub/MPNN_affinity_maturation/flowpacker


In [0]:
#@title 3. Score and filter binders for pTM and ipTM
# boltz score?
from utilities import filter_scores

# ProtenixScore
!python -m protenixscore score --input ./test/flowpacker_out/ --output ./test/ --recursive

# Filter scores
filter_scores("summary.csv", "test/filtered_pdb", threshold_pTM = 0.5, threshold_ipTM = 0.5)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-16c7f3e3-9308-473d-b68f-561c46f53da4/bin/python: No module named protenixscore
python: can't open file '/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation/filter_scores.py': [Errno 2] No such file or directory


In [0]:
#@title 4. Rosetta interaction energy

#pdb_folder = "test"
pdb_file = "test/4dn4_ccl2_8_4.pdb"
interface_dist = 10
energy_filter = -5

!python per_residue_energy_pyrosetta.py --pdb $pdb_file --binder_id A --target_id M --interface_dist 10 --output_dir test/

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu 2026.06+release.e5a76a2dbd7e42e1271d0af460926c86e29e59d1 2026-02-02T18:21:37] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags 
core.init: Rosetta version: PyRosetta4.Release.python312.ubuntu r425 2026.06+release.e5a76a2dbd e

In [0]:
#@title 5. Concatenate key residues

from utilities import process_pdb_mutation_and_renumber
# merge key residues into one pdb file

# Extract Sequences
!python demo_scripts/pdb2fa.py test/af3score/filtered_links csv test/af3score/filtered_links.csv

process_pdb_mutation_and_renumber("test/fixed_residue.csv", "test/merge_motif_pdb/", 
    renumber_chain='M', start_index=9) # The residue numbering of the target chain is consistent with the input target file